# Movie Reviews Embeddings: Static vs Contextual


In [ ]:
!pip install nltk gensim transformers torch


In [ ]:
import re
import os
import numpy as np
import pandas as pd
import nltk

from nltk.corpus import stopwords, movie_reviews
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('movie_reviews')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
documents = []

for fileid in movie_reviews.fileids():
    documents.append({
        'review': movie_reviews.raw(fileid),
        'sentiment': movie_reviews.categories(fileid)[0]
    })

df = pd.DataFrame(documents)

print('Number of reviews:', len(df))
print('\nSentiment distribution:')
print(df['sentiment'].value_counts())

df.head()

Number of reviews: 2000

Sentiment distribution:
sentiment
neg    1000
pos    1000
Name: count, dtype: int64


,review,sentiment
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg


## Text Preprocessing with NLTK

The preprocessing phase performs:

1. Lowercasing
2. Removing URLs
3. Removing mentions
4. Word tokenization using NLTK
5. Removing punctuation/non-alphabetic tokens
6. Removing English stop words
7. Stemming using `PorterStemmer`
8. Lemmatization using `WordNetLemmatizer`


In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    tokens = word_tokenize(text)
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return tokens


df['tokens'] = df['review'].apply(preprocess_text)

df['stemmed_tokens'] = df['tokens'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)

df['lemmatized_tokens'] = df['tokens'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

df[['review', 'sentiment', 'tokens', 'stemmed_tokens', 'lemmatized_tokens']].head()


,review,sentiment,tokens,stemmed_tokens,lemmatized_tokens
0,"plot : two teen couples go to a church party ,...",neg,"[plot, two, teen, couples, go, church, party, ...","[plot, two, teen, coupl, go, church, parti, dr...","[plot, two, teen, couple, go, church, party, d..."
1,the happy bastard's quick movie review \ndamn ...,neg,"[happy, bastard, quick, movie, review, damn, b...","[happi, bastard, quick, movi, review, damn, bu...","[happy, bastard, quick, movie, review, damn, b..."
2,it is movies like these that make a jaded movi...,neg,"[movies, like, make, jaded, movie, viewer, tha...","[movi, like, make, jade, movi, viewer, thank, ...","[movie, like, make, jaded, movie, viewer, than..."
3,""" quest for camelot "" is warner bros . ' firs...",neg,"[quest, camelot, warner, bros, first, attempt,...","[quest, camelot, warner, bro, first, attempt, ...","[quest, camelot, warner, bros, first, attempt,..."
4,synopsis : a mentally unstable man undergoing ...,neg,"[synopsis, mentally, unstable, man, undergoing...","[synopsi, mental, unstabl, man, undergo, psych...","[synopsis, mentally, unstable, man, undergoing..."


## Inspect Original vs Cleaned Reviews

In [ ]:
for _, row in df.head(5).iterrows():
    print('Sentiment:', row['sentiment'])
    print('Original   :', row['review'][:300], '...')
    print('Tokens     :', row['tokens'][:20])
    print('Stemmed    :', row['stemmed_tokens'][:20])
    print('Lemmatized :', row['lemmatized_tokens'][:20])
    print('-' * 80)

Sentiment: neg
Original   : plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the ...
Tokens     : ['plot', 'two', 'teen', 'couples', 'go', 'church', 'party', 'drink', 'drive', 'get', 'accident', 'one', 'guys', 'dies', 'girlfriend', 'continues', 'see', 'life', 'nightmares', 'deal']
Stemmed    : ['plot', 'two', 'teen', 'coupl', 'go', 'church', 'parti', 'drink', 'drive', 'get', 'accid', 'one', 'guy', 'die', 'girlfriend', 'continu', 'see', 'life', 'nightmar', 'deal']
Lemmatized : ['plot', 'two', 'teen', 'couple', 'go', 'church', 'party', 'drink', 'drive', 'get', 'accident', 'one', 'guy', 'dy', 'girlfriend', 'continues', 'see', 'life', 'nightmare', 'deal']
--------------------------------------------------------------------------------
Sentiment: 

##  Prepare Corpus for Embeddings

In [ ]:
tokenized_reviews = df['lemmatized_tokens'].tolist()

print('Number of reviews:', len(tokenized_reviews))
print('First cleaned review:', tokenized_reviews[0][:30])

Number of reviews: 2000
First cleaned review: ['plot', 'two', 'teen', 'couple', 'go', 'church', 'party', 'drink', 'drive', 'get', 'accident', 'one', 'guy', 'dy', 'girlfriend', 'continues', 'see', 'life', 'nightmare', 'deal', 'watch', 'movie', 'sorta', 'find', 'critique', 'movie', 'teen', 'generation', 'touch', 'cool']


In [ ]:
for i in range(3):
    print('Original     :', df.loc[i, 'tokens'][:20])
    print('Stemmed      :', df.loc[i, 'stemmed_tokens'][:20])
    print('Lemmatized   :', df.loc[i, 'lemmatized_tokens'][:20])
    print('-' * 80)


Original     : ['plot', 'two', 'teen', 'couples', 'go', 'church', 'party', 'drink', 'drive', 'get', 'accident', 'one', 'guys', 'dies', 'girlfriend', 'continues', 'see', 'life', 'nightmares', 'deal']
Stemmed      : ['plot', 'two', 'teen', 'coupl', 'go', 'church', 'parti', 'drink', 'drive', 'get', 'accid', 'one', 'guy', 'die', 'girlfriend', 'continu', 'see', 'life', 'nightmar', 'deal']
Lemmatized   : ['plot', 'two', 'teen', 'couple', 'go', 'church', 'party', 'drink', 'drive', 'get', 'accident', 'one', 'guy', 'dy', 'girlfriend', 'continues', 'see', 'life', 'nightmare', 'deal']
--------------------------------------------------------------------------------
Original     : ['happy', 'bastard', 'quick', 'movie', 'review', 'damn', 'bug', 'got', 'head', 'start', 'movie', 'starring', 'jamie', 'lee', 'curtis', 'another', 'baldwin', 'brother', 'william', 'time']
Stemmed      : ['happi', 'bastard', 'quick', 'movi', 'review', 'damn', 'bug', 'got', 'head', 'start', 'movi', 'star', 'jami', 'lee', 'cu

## Static Embedding — Word2Vec


In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=20
)

word = 'good'
vector = w2v_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

print('\nSimilar words:')
print(w2v_model.wv.most_similar(word, topn=5))

Word: good
Vector shape: (100,)
First 10 values: [ 0.15410616  0.21951857  0.1008582   0.12525687  0.21114403 -0.49139488
 -0.05910055  0.6707968   0.26522735 -0.0180788 ]

Similar words:
[('decent', 0.6607694029808044), ('nadir', 0.6294884085655212), ('patinkin', 0.6253750920295715), ('overcomes', 0.6185183525085449), ('paled', 0.6167385578155518)]


## Static Embedding — GloVe


In [ ]:
def load_glove(glove_path):
    embeddings = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.rstrip().split(' ')
            word = values[0]
            vector = np.asarray(values[1:], dtype=np.float32)
            embeddings[word] = vector
    return embeddings

glove_path = 'glove.6B.100d.txt'

if os.path.exists(glove_path):
    glove = load_glove(glove_path)
    word = 'good'
    if word in glove:
        print('Word:', word)
        print('Vector shape:', glove[word].shape)
        print('First 10 values:', glove[word][:10])
    else:
        print('Word not found in GloVe vocabulary.')
else:
    print('GloVe file not found.')
    print('Place glove.6B.100d.txt in the notebook directory to run this section.')

GloVe file not found.
Place glove.6B.100d.txt in the notebook directory to run this section.


## Static Embedding — FastText


In [ ]:
from gensim.models import FastText

fasttext_model = FastText(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

fasttext_model.build_vocab(corpus_iterable=tokenized_reviews)
fasttext_model.train(
    corpus_iterable=tokenized_reviews,
    total_examples=len(tokenized_reviews),
    epochs=20
)

word = 'good'
vector = fasttext_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

unknown_word = 'amazzzzing'
unknown_vector = fasttext_model.wv[unknown_word]
print('\nFastText can generate a vector for:', unknown_word)
print('Vector shape:', unknown_vector.shape)

Word: good
Vector shape: (100,)
First 10 values: [-0.58394533  0.21302453  0.01004412 -0.07439403 -0.33223295  0.11003327
 -0.10998884  0.40684545  0.32349584 -0.1188077 ]

FastText can generate a vector for: amazzzzing
Vector shape: (100,)


## Contextual Embedding — BERT



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.eval()

contextual_reviews = [
    'The movie was excellent and the acting was brilliant.',
    'The movie was terrible and the acting was disappointing.'
]

encoded = bert_tokenizer(
    contextual_reviews,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

with torch.no_grad():
    outputs = bert_model(**encoded)

token_embeddings = outputs.last_hidden_state

print('Token embedding shape:', token_embeddings.shape)
print('BERT hidden size:', token_embeddings.shape[-1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token embedding shape: torch.Size([2, 12, 768])
BERT hidden size: 768


## BERT — Review-Level Embedding with Mean Pooling

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

bert_review_embeddings = mean_pool(
    token_embeddings,
    encoded['attention_mask']
)

print('Review embedding shape:', bert_review_embeddings.shape)
print('First review embedding, first 10 values:')
print(bert_review_embeddings[0][:10].numpy())

Review embedding shape: torch.Size([2, 768])
First review embedding, first 10 values:
[ 0.07255354 -0.2867569  -0.3487154   0.16466726  0.11373287 -0.16831978
  0.23935308  0.9901273   0.23035799 -0.07905494]


## Contextual Embedding — GPT-2


In [ ]:
from transformers import AutoTokenizer, AutoModel

gpt_tokenizer = AutoTokenizer.from_pretrained('gpt2')
gpt_model = AutoModel.from_pretrained('gpt2')
gpt_model.eval()
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_encoded = gpt_tokenizer(
    contextual_reviews,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

with torch.no_grad():
    gpt_outputs = gpt_model(**gpt_encoded)

gpt_token_embeddings = gpt_outputs.last_hidden_state

gpt_review_embeddings = mean_pool(
    gpt_token_embeddings,
    gpt_encoded['attention_mask']
)

print('GPT-2 token embedding shape:', gpt_token_embeddings.shape)
print('GPT-2 review embedding shape:', gpt_review_embeddings.shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 token embedding shape: torch.Size([2, 10, 768])
GPT-2 review embedding shape: torch.Size([2, 768])


In [ ]:
np.save('bert_review_embeddings.npy', bert_review_embeddings.numpy())
np.save('gpt2_review_embeddings.npy', gpt_review_embeddings.numpy())

print('Saved: bert_review_embeddings.npy')
print('Saved: gpt2_review_embeddings.npy')

Saved: bert_review_embeddings.npy
Saved: gpt2_review_embeddings.npy
